# Stromal/Vascular Reintegration v1.3 — scArches-Compatible (P0/P1 Patched)
**Author:** r2end | **Date:** 2026-03-08

## Changes vs v1.2
- [P0-1] Step 16 — batch key rename moved BEFORE `prepare_query_anndata()` (was after → silent bug)
- [P0-2] Step 16 — removed no-op `normalize_total(..., inplace=False)` that had zero effect
- [P1-3] Config  — `PULMONARY_ALVEOLAR_CLUSTERS` now explicitly includes `Fibro_adventitial_c2`
- [P1-4] Step 11 — `SCANVI_TRAINED` flag propagated to Step 14 manifest + Step 16 guard
- [P1-5] Step 5  — `.raw.X` integer-check before counts recovery (refuse log1p silently fed to scVI)
- [P2-6] Steps 8/11 — dry-run validation uses 2k-cell subset instead of full `adata.copy()`

**All v1.2 changes retained** (encode_covariates, prepare_query_anndata, SCANVI var_names.csv, manifest, run log).

## Imports + Environment + Configuration + Annotation Table

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Stromal/Vascular Reintegration: Contamination Removal -> scVI -> Tissue-aware scANVI
======================================================================================
Author: r2end  |  Date: 2026-03-08  |  Version: v1.2 (scArches-Compatible)

scArches Compatibility Notes:
  [A] encode_covariates=True  -- exposes batch embedding to encoder so surgery
      can graft new-batch adapters without touching decoder weights
  [B/C] prepare_query_anndata() -- validates that the saved model can accept
        future query datasets; raises early if gene list is inconsistent
  [D] var_names.csv saved for BOTH scVI and scANVI model directories
  [E] reference_manifest.json -- machine-readable descriptor for query pipelines
  [F] Step 15 -- drop-in query mapping template for new cohorts / centers
"""

import os
for _k in ["OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS","NUMEXPR_NUM_THREADS"]:
    os.environ[_k] = "8"
import matplotlib; matplotlib.use('Agg')
import warnings; warnings.filterwarnings('ignore')
import gc, time, json, numpy as np, pandas as pd
import importlib, sympy

try:
    if not hasattr(sympy, 'printing'):
        sympy.printing = importlib.import_module('sympy.printing')
except Exception as e:
    raise RuntimeError(
        f"Sympy import is broken ({getattr(sympy, '__file__', 'unknown')}). "
        "Ensure real sympy is available and no local sympy.py shadows it."
    ) from e

import scanpy as sc
import scvi
import torch
import matplotlib.pyplot as plt
from scipy import sparse
from pathlib import Path
from sklearn.neighbors import NearestNeighbors

PIPELINE_START = time.time()

# ============================================================================
# CONFIGURATION  --  UPDATE TISSUE_KEY / LUNG_TRACHEA_VALUES BEFORE RUNNING
# ============================================================================

INPUT_H5AD  = Path("/home/h2048/data/py/0120/stromal_analysis_unified/results/"
                   "subcluster_unified_v2_20260128/"
                   "adata_stromal_subclustered_FINAL_v2_20260128.h5ad")
OUTPUT_DIR  = Path("/home/h2048/data/py/0308/stromal_reintegration_v1_2")
FIG_DIR     = OUTPUT_DIR / "figures"
MODEL_DIR   = OUTPUT_DIR / "models"
OUTPUT_H5AD = OUTPUT_DIR / "stromal_reintegrated_scvi_scanvi_v1_2.h5ad"
for d in [OUTPUT_DIR, FIG_DIR, MODEL_DIR]: d.mkdir(parents=True, exist_ok=True)

CELLTYPE_L2 = 'cell_type_L2'
CELLTYPE_L3 = 'cell_type_L3'
BATCH_KEY   = 'sample'

TISSUE_KEY  = 'tissue'
LUNG_TRACHEA_VALUES = {'lung','trachea','Lung','Trachea',
                       'Lung tissue','Tracheal tissue',
                       'lung_tissue','trachea_tissue'}

N_HVG=4000; N_LATENT=75; N_HIDDEN=128; N_LAYERS=2; DROPOUT=0.1
MAX_EPOCHS_SCVI=400; MAX_EPOCHS_SCANVI=200; BATCH_SIZE=256
RANDOM_SEED=42; UNLABELED='Unknown'; MIN_CELLS_BATCH=3
PURITY_K=30; PURITY_THRESHOLD=0.5

# ============================================================================
# REPRODUCIBILITY
# ============================================================================
np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_SEED)
scvi.settings.seed=RANDOM_SEED; scvi.settings.dl_num_workers=0
USE_GPU = torch.cuda.is_available()
sc.settings.verbosity=2; sc.settings.n_jobs=16

# ============================================================================
# ANNOTATION TABLE
# ============================================================================

ANNOTATION_TABLE = [
    # --- Lymphatic Endothelium ---
    ("Endothelia_Lymphatic_c0","Lymphatic Endothelia","Quiescent Lymphatic Endothelia","High","KEEP"),
    ("Endothelia_Lymphatic_c1","Lymphatic Endothelia","Venous-like Lymphatic Endothelia","Medium","REVIEW"),
    ("Endothelia_Lymphatic_c2","Lymphatic Endothelia","Quiescent Lymphatic Endothelia","High","KEEP"),
    # --- Capillary aCap ---
    ("Endothelia_vascular_Cap_a_c0","Contaminants","Interferon-activated Immune Contaminants","High","DROP"),
    ("Endothelia_vascular_Cap_a_c1","Capillary Endothelia","Quiescent Aerocyte-like Capillary Endothelia","High","KEEP"),
    ("Endothelia_vascular_Cap_a_c2","Capillary Endothelia","Quiescent Capillary Endothelia","Medium","KEEP"),
    # --- Capillary gCap ---
    ("Endothelia_vascular_Cap_g_c0","Capillary Endothelia","Unresolved Capillary Endothelia","Low","REVIEW"),
    ("Endothelia_vascular_Cap_g_c1","Capillary Endothelia","Interferon-primed Antigen-presenting Capillary Endothelia","Medium","KEEP"),
    ("Endothelia_vascular_Cap_g_c2","Capillary Endothelia","Unresolved Capillary Endothelia","Low","REVIEW"),
    ("Endothelia_vascular_Cap_g_c3","Capillary Endothelia","LYVE1+ Immune-interacting Capillary Endothelia","Medium","REVIEW"),
    # --- Arterial pulmonary ---
    ("Endothelia_vascular_arterial_pulmonary_c0","Arterial Endothelia","Quiescent Arterial Endothelia","High","KEEP"),
    ("Endothelia_vascular_arterial_pulmonary_c1","Arterial Endothelia","Immunomodulatory Arterial Endothelia","High","KEEP"),
    ("Endothelia_vascular_arterial_pulmonary_c2","Arterial Endothelia","Unresolved Arterial Endothelia","Low","REVIEW"),
    # --- Arterial systemic ---
    ("Endothelia_vascular_arterial_systemic_c0","Arterial Endothelia","Quiescent Arterial Endothelia","High","KEEP"),
    ("Endothelia_vascular_arterial_systemic_c1","Arterial Endothelia","Unresolved Arterial Endothelia","Low","REVIEW"),
    # --- Venous pulmonary ---
    ("Endothelia_vascular_venous_pulmonary_c0","Venous Endothelia","Homeostatic Venous Endothelia","Medium","KEEP"),
    ("Endothelia_vascular_venous_pulmonary_c1","Venous Endothelia","Unresolved Venous Endothelia","Low","REVIEW"),
    ("Endothelia_vascular_venous_pulmonary_c2","Venous Endothelia","Unresolved Venous Endothelia","Low","REVIEW"),
    # --- Venous systemic ---
    ("Endothelia_vascular_venous_systemic_c0","Venous Endothelia","Activated Antigen-presenting Venous Endothelia","Medium","REVIEW"),
    ("Endothelia_vascular_venous_systemic_c1","Venous Endothelia","Immune-recruiting (ACKR1+SELE+) Venous Endothelia","High","KEEP"),
    ("Endothelia_vascular_venous_systemic_c2","Venous Endothelia","Angiogenic Venous Endothelia","High","KEEP"),
    ("Endothelia_vascular_venous_systemic_c3","Contaminants","MHC-II-high APC-like Contaminants","High","DROP"),
    ("Endothelia_vascular_venous_systemic_c4","Fibroblasts","Stress-activated Fibroblasts","Medium","REASSIGN_TO_FIBRO"),
    ("Endothelia_vascular_venous_systemic_c5","Contaminants","Plasma/B-cell Contaminants","High","DROP"),
    # --- Fibroblast adventitial ---
    ("Fibro_adventitial_c0","Adventitial Fibroblasts","Perivascular Adventitial Fibroblasts","High","KEEP"),
    ("Fibro_adventitial_c1","Adventitial Fibroblasts","Niche-supporting Adventitial Fibroblasts","High","KEEP"),
    ("Fibro_adventitial_c2","Alveolar Fibroblasts","Homeostatic Alveolar Fibroblasts","Medium","REASSIGN_TO_ALVEOLAR"),
    # --- Fibroblast alveolar ---
    ("Fibro_alveolar_c0","Alveolar Fibroblasts","Unresolved Alveolar Fibroblasts","Low","REVIEW"),
    ("Fibro_alveolar_c1","Alveolar Fibroblasts","Activated ECM-high Alveolar Fibroblasts","High","KEEP"),
    ("Fibro_alveolar_c2","Alveolar Fibroblasts","Homeostatic Alveolar Fibroblasts","High","KEEP"),
    # --- Fibroblast myofibroblast ---
    ("Fibro_myofibroblast_c0","Contaminants","Osteogenic Contaminants","High","DROP"),
    ("Fibro_myofibroblast_c1","Myofibroblasts","Unresolved Myofibroblasts","Low","REVIEW"),
    # --- Fibroblast peribronchial ---
    ("Fibro_peribronchial_c0","Peribronchial Fibroblasts","Lipogenic Peribronchial Fibroblasts","High","KEEP"),
    ("Fibro_peribronchial_c1","Peribronchial Fibroblasts","Developmental-signaling Peribronchial Fibroblasts","High","KEEP"),
    ("Fibro_peribronchial_c2","Neural/Glial Stromal","Neural-like Stromal Cells","Medium","REVIEW"),
    # --- Pericyte pulmonary ---
    ("Muscle_pericyte_pulmonary_c0","Pericytes","Quiescent Pulmonary Pericytes","High","KEEP"),
    ("Muscle_pericyte_pulmonary_c1","Pericytes","Unresolved Pulmonary Pericytes","Low","REVIEW"),
    ("Muscle_pericyte_pulmonary_c2","Pericytes","Contractile Pulmonary Pericytes","High","KEEP"),
    ("Muscle_pericyte_pulmonary_c3","Pericytes","Precursor Pulmonary Pericytes","Medium","KEEP"),
    # --- Pericyte systemic ---
    ("Muscle_pericyte_systemic_c0","Pericytes","Quiescent Systemic Pericytes","High","KEEP"),
    ("Muscle_pericyte_systemic_c1","Pericytes","Contractile Systemic Pericytes","High","KEEP"),
    # --- Perivascular immune-recruiting ---
    ("Muscle_perivascular_immune_recruiting_c0","Vascular SMC","Contractile Vascular Smooth Muscle Cells","High","KEEP"),
    ("Muscle_perivascular_immune_recruiting_c1","Pericytes","Quiescent Systemic Pericytes","Medium","REVIEW"),
    # --- Smooth muscle ---
    ("Muscle_smooth_arterial_systemic_c0","Airway SMC","Contractile Airway Smooth Muscle Cells","High","KEEP"),
    ("Muscle_smooth_arterial_systemic_c1","Vascular SMC","Contractile Vascular Smooth Muscle Cells","High","KEEP"),
    ("Muscle_smooth_pulmonary_c0","Smooth Muscle Cells","Unresolved Smooth Muscle Cells","Low","REVIEW"),
    ("Muscle_smooth_pulmonary_c1","Smooth Muscle Cells","Contractile Smooth Muscle Cells","High","KEEP"),
    # --- Schwann ---
    ("Schwann_nonmyelinating_c0","Schwann Cells","Non-myelinating Schwann Cells","High","KEEP"),
]

annot_df = pd.DataFrame(ANNOTATION_TABLE,
    columns=['cluster','L2_new','coarse_L3','confidence','action']
).set_index('cluster')

DROP_CLUSTERS = annot_df[annot_df['action']=='DROP'].index.tolist()

# [P1-3] Explicit inclusion of Fibro_adventitial_c2:
# Its cluster NAME does not contain 'alveolar', but it is REASSIGNED to
# 'Alveolar Fibroblasts' in Step 3. Without this explicit entry it would
# escape the tissue-aware Unknown filter.
PULMONARY_ALVEOLAR_CLUSTERS = sorted(set(
    annot_df.index[
        annot_df.index.str.contains('pulmonary|alveolar', case=False, regex=True)
    ].tolist()
    + ['Fibro_adventitial_c2']  # reassigned to Alveolar Fibroblasts in Step 3
))

print(f"\nDROP clusters ({len(DROP_CLUSTERS)}):")
for c in DROP_CLUSTERS: print(f"  {c} -> [{annot_df.loc[c,'coarse_L3']}]")
print(f"\nPulmonary/Alveolar clusters for tissue-aware scANVI ({len(PULMONARY_ALVEOLAR_CLUSTERS)}):")
for c in PULMONARY_ALVEOLAR_CLUSTERS: print(f"  {c}")


Seed set to 42



DROP clusters (4):
  Endothelia_vascular_Cap_a_c0 -> [Interferon-activated Immune Contaminants]
  Endothelia_vascular_venous_systemic_c3 -> [MHC-II-high APC-like Contaminants]
  Endothelia_vascular_venous_systemic_c5 -> [Plasma/B-cell Contaminants]
  Fibro_myofibroblast_c0 -> [Osteogenic Contaminants]

Pulmonary/Alveolar clusters for tissue-aware scANVI (16):
  Endothelia_vascular_arterial_pulmonary_c0
  Endothelia_vascular_arterial_pulmonary_c1
  Endothelia_vascular_arterial_pulmonary_c2
  Endothelia_vascular_venous_pulmonary_c0
  Endothelia_vascular_venous_pulmonary_c1
  Endothelia_vascular_venous_pulmonary_c2
  Fibro_adventitial_c2
  Fibro_alveolar_c0
  Fibro_alveolar_c1
  Fibro_alveolar_c2
  Muscle_pericyte_pulmonary_c0
  Muscle_pericyte_pulmonary_c1
  Muscle_pericyte_pulmonary_c2
  Muscle_pericyte_pulmonary_c3
  Muscle_smooth_pulmonary_c0
  Muscle_smooth_pulmonary_c1


## Step 1: LOAD DATA

In [2]:
# ============================================================================
# STEP 1: LOAD DATA
# ============================================================================
print("\n" + "="*70 + "\nSTEP 1: LOAD DATA\n" + "="*70)
adata = sc.read_h5ad(INPUT_H5AD)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"Obs keys: {list(adata.obs.columns)}")

for col in [CELLTYPE_L2, BATCH_KEY]:
    assert col in adata.obs.columns, f"[ERROR] Missing: {col}"
assert CELLTYPE_L3 in adata.obs.columns, f"[ERROR] Missing configured L3 column: {CELLTYPE_L3}"

def _annotation_match_count(values):
    return pd.Index(values).isin(annot_df.index).sum()

l3_str = adata.obs[CELLTYPE_L3].astype(str)
l3_unique = l3_str.unique()
n_match = _annotation_match_count(l3_unique)

best_col = CELLTYPE_L3
best_unique = l3_unique
best_match = n_match

candidate_cols = []
if 'subcluster_id' in adata.obs.columns:
    candidate_cols.append('subcluster_id')
candidate_cols.extend([
    c for c in adata.obs.columns
    if c != CELLTYPE_L3 and ('l3' in c.lower() or 'subcluster' in c.lower())
])

for c in dict.fromkeys(candidate_cols):
    vals = adata.obs[c].astype(str).unique()
    m = _annotation_match_count(vals)
    if m > best_match:
        best_col = c; best_unique = vals; best_match = m

if best_match == 0 and CELLTYPE_L2 in adata.obs.columns and 'subcluster_id' in adata.obs.columns:
    reconstructed_col = '__reconstructed_cluster_id'
    reconstructed = (
        adata.obs[CELLTYPE_L2].astype(str)
        + '_c'
        + adata.obs['subcluster_id'].astype(str)
    )
    adata.obs[reconstructed_col] = pd.Categorical(reconstructed)
    vals = adata.obs[reconstructed_col].astype(str).unique()
    m = _annotation_match_count(vals)
    if m > best_match:
        print(f"[INFO] Reconstructed cluster IDs -> '{reconstructed_col}' (coverage {m}/{len(vals)})")
        best_col = reconstructed_col; best_unique = vals; best_match = m

if best_col != CELLTYPE_L3 and best_match > 0:
    print(f"[INFO] Auto-switch CELLTYPE_L3: '{CELLTYPE_L3}' -> '{best_col}' (coverage {best_match}/{len(best_unique)})")
    CELLTYPE_L3 = best_col; l3_unique = best_unique; n_match = best_match
else:
    n_match = best_match

TISSUE_KEY_AVAILABLE = TISSUE_KEY in adata.obs.columns
if not TISSUE_KEY_AVAILABLE:
    similar = [c for c in adata.obs.columns if any(k in c.lower() for k in ('tissue','organ','site'))]
    print(f"[WARN] TISSUE_KEY='{TISSUE_KEY}' not found. Candidates: {similar}")
    print(f"       Tissue-aware Unknown assignment will be SKIPPED.")
else:
    print(f"\nTissue distribution:\n{adata.obs[TISSUE_KEY].value_counts().to_string()}")

not_in_table = [x for x in l3_unique if x not in annot_df.index]
print(f"Annotation table coverage: {len(l3_unique)-len(not_in_table)}/{len(l3_unique)}")
if not_in_table:
    print(f"  [WARN] Unmatched: {not_in_table[:20]}")
if n_match == 0:
    raise ValueError(f"[ERROR] No values in '{CELLTYPE_L3}' match ANNOTATION_TABLE.")



STEP 1: LOAD DATA
Loaded: 60,828 cells x 36,789 genes
Obs keys: ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'plateID', 'status', 'donorID', 'cDate', 'age', 'sex', 'cellType', 'percent.mt', 'percent.ribo', 'tissue', 'tissue_sampling_method', 'dataset', 'sample', 'percent.rb', 'decontX_contamination', 'decontX_clusters', 'nCount_decontXcounts', 'nFeature_decontXcounts', 'donor_id', 'Group', 'Ethnicity_inferred', 'Smoker', 'COVID_status', 'First_symptoms_collection_interval', 'Kit_version', 'batch', 'log1p_n_genes', 'percent_total_sarscov2', 'n_counts_sarscov2', 'scrublet_score', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'Source', 'Location', 'CellType', 'BroadCellType', 'organism_ontology_term_id', 'BMI', 'age_or_mean

## Step 2: ATTACH ANNOTATION METADATA

In [3]:
# ============================================================================
# STEP 2: ATTACH ANNOTATION TABLE TO adata.obs
# ============================================================================
print("\n" + "="*70 + "\nSTEP 2: ATTACH ANNOTATION TABLE TO adata.obs\n" + "="*70)
l3_str = adata.obs[CELLTYPE_L3].astype(str)
coarse_mapped     = l3_str.map(annot_df['coarse_L3'])
action_mapped     = l3_str.map(annot_df['action'])
confidence_mapped = l3_str.map(annot_df['confidence'])

n_mapped = coarse_mapped.notna().sum()
print(f"Annotation-mapped cells: {n_mapped:,}/{adata.n_obs:,}")
if n_mapped == 0:
    raise ValueError("[ERROR] Annotation mapping produced 0 matched cells.")

adata.obs['coarse_L3']        = coarse_mapped.fillna('Unresolved')
adata.obs['annot_action']     = action_mapped.fillna('REVIEW')
adata.obs['annot_confidence'] = confidence_mapped.fillna('Low')
print(adata.obs['annot_action'].value_counts().to_string())

if n_mapped < adata.n_obs:
    unmatched = pd.Index(l3_str[coarse_mapped.isna()].unique()).tolist()
    print(f"[WARN] Unmatched {CELLTYPE_L3} values (first 20): {unmatched[:20]}")



STEP 2: ATTACH ANNOTATION TABLE TO adata.obs
Annotation-mapped cells: 60,828/60,828
annot_action
KEEP                    32717
REVIEW                  18742
DROP                     4134
REASSIGN_TO_ALVEOLAR     3558
REASSIGN_TO_FIBRO        1677


## Step 3: REASSIGN MISCLASSIFIED CLUSTERS

In [4]:
# ============================================================================
# STEP 3: REASSIGN MISCLASSIFIED CLUSTERS
# ============================================================================
print("\n" + "="*70 + "\nSTEP 3: REASSIGN MISCLASSIFIED CLUSTERS\n" + "="*70)
l3_str = adata.obs[CELLTYPE_L3].astype(str)

if isinstance(adata.obs[CELLTYPE_L2].dtype, pd.CategoricalDtype):
    missing_cats = [x for x in ['Fibroblasts', 'Alveolar Fibroblasts']
                    if x not in adata.obs[CELLTYPE_L2].cat.categories]
    if missing_cats:
        adata.obs[CELLTYPE_L2] = adata.obs[CELLTYPE_L2].cat.add_categories(missing_cats)

mask_fibro = l3_str == 'Endothelia_vascular_venous_systemic_c4'
adata.obs.loc[mask_fibro, CELLTYPE_L2] = 'Fibroblasts'
print(f"REASSIGN_TO_FIBRO: {mask_fibro.sum():,} cells -> L2='Fibroblasts'")

mask_alv = l3_str == 'Fibro_adventitial_c2'
adata.obs.loc[mask_alv, CELLTYPE_L2] = 'Alveolar Fibroblasts'
print(f"REASSIGN_TO_ALVEOLAR: {mask_alv.sum():,} cells -> L2='Alveolar Fibroblasts'")



STEP 3: REASSIGN MISCLASSIFIED CLUSTERS
REASSIGN_TO_FIBRO: 1,677 cells -> L2='Fibroblasts'
REASSIGN_TO_ALVEOLAR: 3,558 cells -> L2='Alveolar Fibroblasts'


## Step 4: REMOVE CONTAMINATION CLUSTERS

In [5]:
# ============================================================================
# STEP 4: REMOVE CONTAMINATION CLUSTERS
# ============================================================================
print("\n" + "="*70 + "\nSTEP 4: REMOVE CONTAMINATION CLUSTERS\n" + "="*70)
n_before  = adata.n_obs
drop_mask = adata.obs[CELLTYPE_L3].astype(str).isin(DROP_CLUSTERS)
for c in DROP_CLUSTERS:
    n = (adata.obs[CELLTYPE_L3].astype(str)==c).sum()
    print(f"  {c}: {n:,}  [{annot_df.loc[c,'coarse_L3'] if c in annot_df.index else 'N/A'}]")
adata = adata[~drop_mask].copy()
print(f"{n_before:,} -> {adata.n_obs:,} cells  (removed {n_before-adata.n_obs:,})")
l3_str = adata.obs[CELLTYPE_L3].astype(str)
gc.collect()



STEP 4: REMOVE CONTAMINATION CLUSTERS
  Endothelia_vascular_Cap_a_c0: 265  [Interferon-activated Immune Contaminants]
  Endothelia_vascular_venous_systemic_c3: 2,764  [MHC-II-high APC-like Contaminants]
  Endothelia_vascular_venous_systemic_c5: 705  [Plasma/B-cell Contaminants]
  Fibro_myofibroblast_c0: 400  [Osteogenic Contaminants]
60,828 -> 56,694 cells  (removed 4,134)


6420

## Step 5: VERIFY DATA LAYERS

In [6]:
# ============================================================================
# STEP 5: VERIFY DATA LAYERS
# ============================================================================
print("\n" + "="*70 + "\nSTEP 5: VERIFY DATA LAYERS\n" + "="*70)

# [P1-5] Helper: refuse to silently feed log1p data as counts to scVI.
# .raw.X may contain log1p normalized expression in many AnnData objects.
def _looks_like_counts(x, atol=1e-6, max_check=100_000):
    """Return True if x values appear to be non-negative integers (raw counts)."""
    if sparse.issparse(x):
        vals = x.data
    else:
        vals = np.ravel(x)
    if vals.size == 0:
        return True
    vals = vals[:min(max_check, vals.size)]
    return np.all(vals >= 0) and np.allclose(vals, np.round(vals), atol=atol)

if 'counts' not in adata.layers:
    if adata.raw is None:
        raise ValueError("No 'counts' layer and no .raw -- cannot continue.")
    print("[WARN] 'counts' layer missing -- attempting recovery from .raw.X")
    missing = adata.var_names.difference(adata.raw.var_names)
    if len(missing) > 0:
        raise ValueError(f".raw is missing {len(missing)} current genes; cannot recover counts safely.")
    raw_x = adata.raw[:, adata.var_names].X
    if not _looks_like_counts(raw_x):
        raise ValueError(
            "[ERROR] .raw.X does not look like raw integer counts (may be log1p). "
            "Refusing to feed it to scVI as counts. Please set adata.layers['counts'] explicitly."
        )
    adata.layers['counts'] = sparse.csr_matrix(raw_x).astype(np.float32)
    print("[OK] counts recovered from .raw.X (integer check passed)")

if 'log1p' not in adata.layers:
    adata.layers['log1p'] = adata.X.copy()
adata.X = adata.layers['log1p']
for lk in ['counts','log1p']:
    if lk in adata.layers and not sparse.issparse(adata.layers[lk]):
        adata.layers[lk] = sparse.csr_matrix(adata.layers[lk])
if not sparse.issparse(adata.X):
    adata.X = sparse.csr_matrix(adata.X)
print(f"[OK] shape={adata.shape}, counts={adata.layers['counts'].data.nbytes/1e9:.2f} GB")



STEP 5: VERIFY DATA LAYERS
[OK] shape=(56694, 36789), counts=0.98 GB


## Step 6: FILTER SMALL BATCHES

In [7]:
# ============================================================================
# STEP 6: FILTER SMALL BATCHES
# ============================================================================
print("\n" + "="*70 + "\nSTEP 6: FILTER SMALL BATCHES\n" + "="*70)
bc = adata.obs[BATCH_KEY].value_counts()
small = bc[bc < MIN_CELLS_BATCH].index.tolist()
if small:
    n_pre = adata.n_obs
    adata = adata[~adata.obs[BATCH_KEY].isin(small)].copy()
    print(f"Removed {len(small)} small batches: {n_pre:,} -> {adata.n_obs:,}")
else:
    print(f"[OK] All {adata.obs[BATCH_KEY].nunique()} batches pass minimum size")
gc.collect()



STEP 6: FILTER SMALL BATCHES
Removed 3 small batches: 56,694 -> 56,689


6332

## Step 7: HVG SELECTION + FULL-GENE .raw (shared memory, zero extra cost)

In [8]:
# ============================================================================
# STEP 7: HVG SELECTION + PRESERVE .raw (shared memory)
# ============================================================================
print("\n" + "="*70 + "\nSTEP 7: HVG SELECTION + PRESERVE .raw\n" + "="*70)
print(f"Selecting {N_HVG} HVGs from {adata.n_vars:,} genes...")
try:
    sc.pp.highly_variable_genes(adata, layer='counts', n_top_genes=N_HVG,
                                 batch_key=BATCH_KEY, flavor='seurat_v3', subset=False)
    hvg_method = "batch-aware seurat_v3"
except Exception as e:
    print(f"  [WARN] {e}")
    try:
        sc.pp.highly_variable_genes(adata, layer='counts', n_top_genes=N_HVG,
                                     flavor='seurat_v3', subset=False)
        hvg_method = "seurat_v3 (no batch)"
    except Exception as e2:
        print(f"  [WARN] {e2}")
        sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, subset=False)
        hvg_method = "default"
print(f"HVG method: {hvg_method} | HVGs: {adata.var['highly_variable'].sum():,}")

# CRITICAL: preserve full genes to .raw BEFORE subsetting (shared memory, no copy)
print(f"Saving {adata.n_vars:,}-gene data to .raw (shared memory)...")
adata.raw = sc.AnnData(
    X   = adata.layers['counts'],   # shared memory -- no .copy()
    obs = adata.obs.copy(),
    var = adata.var.copy()
)
adata = adata[:, adata.var['highly_variable']].copy()

# FIX [4]: recover counts from .raw not from log1p .X
if 'counts' not in adata.layers:
    print("[WARN] counts layer missing after HVG subset -- recovering from .raw")
    raw_hvg_counts = adata.raw[:, adata.var_names].X
    adata.layers['counts'] = sparse.csr_matrix(raw_hvg_counts).astype(np.float32)

print(f"HVG subset: {adata.shape} | .raw: {adata.raw.n_vars:,} genes (full)")
gc.collect()



STEP 7: HVG SELECTION + PRESERVE .raw
Selecting 4000 HVGs from 36,789 genes...
extracting highly variable genes
  [WARN] b'There are other near singularities as well. 0.090619\n'
extracting highly variable genes
HVG method: seurat_v3 (no batch) | HVGs: 4,000
Saving 36,789-gene data to .raw (shared memory)...
HVG subset: (56689, 4000) | .raw: 36,789 genes (full)


6461

## Step 8: scVI TRAINING (scArches-compatible)

In [9]:
# ============================================================================
# STEP 8: scVI TRAINING  -- scArches-compatible
# ============================================================================
# scArches compatibility requirements:
#   [A] encode_covariates=True
#       The batch covariate is injected into the ENCODER (not just the decoder),
#       so architecture surgery can add new-batch adapters later without touching
#       the shared decoder.  Without this, load_query_data() still works but the
#       latent space may be less separable across new batches.
#   [B] After training, call prepare_query_anndata() on adata itself as a dry-run
#       validation -- confirms that the saved model is query-ready.
# ============================================================================
print("\n" + "="*70 + "\nSTEP 8: scVI TRAINING (scArches-compatible)\n" + "="*70)
SCVI_MODEL_PATH = MODEL_DIR / "scvi_stromal_v1"

scvi.model.SCVI.setup_anndata(adata, layer='counts', batch_key=BATCH_KEY)

# [A] encode_covariates=True is the key scArches-compatibility flag
model_scvi = scvi.model.SCVI(
    adata,
    n_latent=N_LATENT,
    n_hidden=N_HIDDEN,
    n_layers=N_LAYERS,
    dropout_rate=DROPOUT,
    gene_likelihood='nb',
    dispersion='gene-batch',
    encode_covariates=True,      # SCARCHES: batch info flows through encoder
)
print(f"n_latent={N_LATENT}, encode_covariates=True, cells={adata.n_obs:,}, HVGs={adata.n_vars:,}")

train_accelerator = 'gpu' if USE_GPU else 'cpu'
train_devices = 1
print(f"Training backend: accelerator={train_accelerator}, devices={train_devices}")

t0 = time.time()
model_scvi.train(
    max_epochs=MAX_EPOCHS_SCVI,
    batch_size=BATCH_SIZE,
    early_stopping=True,
    early_stopping_patience=20,
    train_size=0.9,
    accelerator=train_accelerator,
    devices=train_devices,
    plan_kwargs={'lr': 1e-3}
)
print(f"[OK] scVI done in {(time.time()-t0)/60:.1f} min")

model_scvi.save(str(SCVI_MODEL_PATH), overwrite=True)

# Save HVG gene list (required by scArches query pipeline to align gene space)
hvg_gene_list = adata.var_names.tolist()
pd.Series(hvg_gene_list).to_csv(SCVI_MODEL_PATH / "var_names.csv", index=False, header=False)
print(f"[OK] var_names.csv saved ({len(hvg_gene_list)} HVGs) -> {SCVI_MODEL_PATH/'var_names.csv'}")

# [B] scArches dry-run validation:
# prepare_query_anndata modifies a copy of adata in-place to match the saved
# model's expected gene order/set.  Running it on the training data confirms
# the model artifact is structurally valid for future query mapping.
print("\n[scArches validation] Running prepare_query_anndata() dry-run...")
try:
    # [P2-6] Use 2k-cell subset to avoid duplicating full adata in RAM
    _n_check = min(2000, adata.n_obs)
    adata_check = adata[:_n_check, :].copy()
    scvi.model.SCVI.prepare_query_anndata(adata_check, str(SCVI_MODEL_PATH))
    del adata_check
    gc.collect()
    print("[OK] scVI model passes scArches prepare_query_anndata() validation")
except Exception as e:
    print(f"[WARN] prepare_query_anndata() validation raised: {e}")
    print("       Check that var_names.csv matches adata.var_names exactly.")

adata.obsm['X_scvi'] = model_scvi.get_latent_representation()
print(f"[OK] X_scvi: {adata.obsm['X_scvi'].shape}")


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



STEP 8: scVI TRAINING (scArches-compatible)
n_latent=75, encode_covariates=True, cells=56,689, HVGs=4,000
Training backend: accelerator=gpu, devices=1


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 400/400: 100%|██████████| 400/400 [38:16<00:00,  5.70s/it, v_num=1, train_loss_step=685, train_loss_epoch=748]

`Trainer.fit` stopped: `max_epochs=400` reached.


Epoch 400/400: 100%|██████████| 400/400 [38:16<00:00,  5.74s/it, v_num=1, train_loss_step=685, train_loss_epoch=748]
[OK] scVI done in 38.3 min
[OK] var_names.csv saved (4000 HVGs) -> /home/h2048/data/py/0308/stromal_reintegration_v1_2/models/scvi_stromal_v1/var_names.csv

[scArches validation] Running prepare_query_anndata() dry-run...
INFO     File /home/h2048/data/py/0308/stromal_reintegration_v1_2/models/scvi_stromal_v1/model.pt already          
         downloaded                                                                                                
INFO     Found 100.0% reference vars in query data.                                                                
[OK] scVI model passes scArches prepare_query_anndata() validation
[OK] X_scvi: (56689, 75)


## Step 9: NEIGHBORS + UMAP (scVI latent)

In [10]:
# ============================================================================
# STEP 9: NEIGHBORS + UMAP (scVI latent)
# ============================================================================
print("\n" + "="*70 + "\nSTEP 9: NEIGHBORS + UMAP (scVI)\n" + "="*70)
sc.pp.neighbors(adata, use_rep='X_scvi', n_neighbors=30,
                random_state=RANDOM_SEED, key_added='neighbors_scvi')
sc.tl.umap(adata, neighbors_key='neighbors_scvi', random_state=RANDOM_SEED)
adata.obsm['X_umap_scvi'] = adata.obsm['X_umap'].copy()
fig, ax = plt.subplots(figsize=(9,7))
sc.pl.embedding(adata, basis='umap', color=CELLTYPE_L2, ax=ax,
                show=False, frameon=False, size=2, legend_loc='right margin',
                title='Post-scVI UMAP (L2)')
fig.savefig(FIG_DIR/'scvi_umap_L2.pdf', dpi=300, bbox_inches='tight')
plt.close('all'); gc.collect()
print("[OK] Saved: scvi_umap_L2.pdf")



STEP 9: NEIGHBORS + UMAP (scVI)
computing neighbors
    finished (0:01:12)
computing UMAP
    finished (0:01:46)
[OK] Saved: scvi_umap_L2.pdf


## Step 10: BUILD scANVI LABELS (tissue-aware coarse_L3)

In [11]:
# ============================================================================
# STEP 10: BUILD scANVI TRAINING LABELS
# ============================================================================
print("\n" + "="*70 + "\nSTEP 10: BUILD scANVI TRAINING LABELS\n" + "="*70)

scanvi_labels = adata.obs['coarse_L3'].astype(str).copy()

if TISSUE_KEY_AVAILABLE:
    tissue_str  = adata.obs[TISSUE_KEY].astype(str)
    tissue_norm = (
        tissue_str.str.strip().str.lower()
        .str.replace(r'[_-]+', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
    )
    lung_trachea_norm = {
        x.strip().lower().replace('_', ' ').replace('-', ' ')
        for x in LUNG_TRACHEA_VALUES
    }
    exact_tissue_match   = tissue_norm.isin(lung_trachea_norm)
    keyword_tissue_match = tissue_norm.str.contains(
        r'\blung\b|\btrachea\b|\bairway\b|\bbronch|\bbronchi|\bparenchyma\b|\bpulmon',
        regex=True, na=False,
    )
    is_lung_or_trachea = exact_tissue_match | keyword_tissue_match

    auto_extra = sorted(tissue_str[keyword_tissue_match & ~exact_tissue_match].unique().tolist())
    if auto_extra:
        print(f"[INFO] Additional tissue values treated as lung/trachea by keyword: {auto_extra[:20]}")

    is_pulm_alv   = adata.obs[CELLTYPE_L3].astype(str).isin(PULMONARY_ALVEOLAR_CLUSTERS)
    tissue_mismatch = is_pulm_alv & ~is_lung_or_trachea
    n_mismatch = tissue_mismatch.sum()
    print(f"Lung/trachea cells:               {is_lung_or_trachea.sum():,}")
    print(f"Pulmonary/alveolar cluster cells: {is_pulm_alv.sum():,}")
    print(f"Tissue mismatch -> Unknown:       {n_mismatch:,}")
    if n_mismatch > 0:
        detail = (adata.obs.loc[tissue_mismatch, [CELLTYPE_L3, TISSUE_KEY]]
                  .value_counts().reset_index(name='n_cells'))
        print(f"\nMismatch detail:\n{detail.head(20).to_string(index=False)}")
    scanvi_labels[tissue_mismatch] = UNLABELED
else:
    print("[WARN] Tissue key unavailable -- skipping tissue-aware filtering.")

# FIX [2]: Vectorized kNN purity (eliminates Python for loop)
print(f"\nComputing kNN purity (k={PURITY_K}) in scVI latent space...")
knn = NearestNeighbors(n_neighbors=PURITY_K, algorithm='auto', n_jobs=16)
knn.fit(adata.obsm['X_scvi'])
_, knn_idx = knn.kneighbors(adata.obsm['X_scvi'])
labels_arr     = scanvi_labels.values
neighbor_labels = labels_arr[knn_idx]
purity = np.mean(neighbor_labels == labels_arr[:, None], axis=1).astype(np.float32)
is_unknown_mask = labels_arr == UNLABELED
purity[is_unknown_mask] = 1.0
adata.obs['label_purity_scanvi'] = purity

low_purity = (purity < PURITY_THRESHOLD) & ~is_unknown_mask
scanvi_labels[low_purity] = UNLABELED
print(f"  Low purity -> Unknown: {low_purity.sum():,} ({low_purity.sum()/adata.n_obs*100:.1f}%)")
del knn, knn_idx, neighbor_labels; gc.collect()

adata.obs['scanvi_label'] = scanvi_labels.values
n_labeled = (scanvi_labels != UNLABELED).sum()
n_unknown  = (scanvi_labels == UNLABELED).sum()
print(f"\nLabel summary: {n_labeled:,} labeled | {n_unknown:,} Unknown")
print("Label distribution:")
for lbl, n in scanvi_labels.value_counts().head(25).items():
    print(f"  {lbl}: {n:,}{'  <-- UNLABELED' if lbl==UNLABELED else ''}")



STEP 10: BUILD scANVI TRAINING LABELS
[INFO] Additional tissue values treated as lung/trachea by keyword: ['lung parenchyma', 'respiratory airway']
Lung/trachea cells:               18,705
Pulmonary/alveolar cluster cells: 10,827
Tissue mismatch -> Unknown:       3,558

Mismatch detail:
__reconstructed_cluster_id tissue  n_cells
      Fibro_adventitial_c2   nose     2231
      Fibro_adventitial_c2  sinus      687
Muscle_smooth_pulmonary_c0   nose      417
Muscle_smooth_pulmonary_c1   nose      212
Muscle_smooth_pulmonary_c1  sinus        7
Muscle_smooth_pulmonary_c0  sinus        4

Computing kNN purity (k=30) in scVI latent space...
  Low purity -> Unknown: 26,366 (46.5%)

Label summary: 26,765 labeled | 29,924 Unknown
Label distribution:
  Unknown: 29,924  <-- UNLABELED
  Activated Antigen-presenting Venous Endothelia: 4,708
  Immune-recruiting (ACKR1+SELE+) Venous Endothelia: 3,972
  Angiogenic Venous Endothelia: 3,032
  Quiescent Lymphatic Endothelia: 2,907
  Unresolved Capillary 

## Step 11: scANVI TRAINING (scArches-compatible)

In [12]:
# ============================================================================
# STEP 11: scANVI TRAINING  -- scArches-compatible
# ============================================================================
# scArches compatibility requirements:
#   Inherits encode_covariates=True from the scVI base model (no extra flag needed).
#   [C] After training, call prepare_query_anndata() dry-run on SCANVI model.
#   [D] Save var_names.csv for the SCANVI model directory as well, so the query
#       pipeline can load SCANVI independently of scVI without path gymnastics.
# ============================================================================
print("\n" + "="*70 + "\nSTEP 11: scANVI TRAINING (scArches-compatible)\n" + "="*70)
SCANVI_MODEL_PATH = MODEL_DIR / "scanvi_stromal_v1"

_labels = adata.obs['scanvi_label'].astype(str)
_labeled_mask = _labels != UNLABELED
_n_labeled_classes = _labels[_labeled_mask].nunique()
print(f"Labeled classes (excluding '{UNLABELED}'): {_n_labeled_classes}")

# [P1-4] SCANVI_TRAINED flag: propagated to manifest (Step 14) and query guard (Step 16)
SCANVI_TRAINED = (_n_labeled_classes >= 2)

if not SCANVI_TRAINED:
    print("[WARN] <2 labeled classes detected; skipping scANVI training and using scVI fallback outputs.")
    adata.obsm['X_scanvi']              = adata.obsm['X_scvi'].copy()
    adata.obs['cell_type_scanvi_pred']  = _labels.copy()
    adata.obs['scanvi_uncertainty']     = np.where(_labels == UNLABELED, 1.0, 0.0).astype(np.float32)
    print(f"[OK] Fallback X_scanvi: {adata.obsm['X_scanvi'].shape}")
    print("Fallback prediction distribution:")
    print(adata.obs['cell_type_scanvi_pred'].value_counts().head(25).to_string())
else:
    # FIX [1]: No redundant setup_anndata before from_scvi_model
    model_scanvi = scvi.model.SCANVI.from_scvi_model(
        model_scvi,
        unlabeled_category=UNLABELED,
        labels_key='scanvi_label'
    )

    # FIX [3]: Release scVI from GPU/CPU before scANVI training
    del model_scvi
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("[OK] GPU cache cleared before scANVI training")

    train_accelerator = 'gpu' if USE_GPU else 'cpu'
    train_devices = 1
    print(f"Training backend: accelerator={train_accelerator}, devices={train_devices}")

    import inspect
    scanvi_train_kwargs = dict(
        max_epochs=MAX_EPOCHS_SCANVI,
        batch_size=BATCH_SIZE,
        early_stopping=True,
        early_stopping_patience=20,
        train_size=0.9,
        accelerator=train_accelerator,
        devices=train_devices,
    )
    if 'n_samples_per_label' in inspect.signature(model_scanvi.train).parameters:
        scanvi_train_kwargs['n_samples_per_label'] = 100

    t0 = time.time()
    model_scanvi.train(**scanvi_train_kwargs)
    print(f"[OK] scANVI done in {(time.time()-t0)/60:.1f} min")
    model_scanvi.save(str(SCANVI_MODEL_PATH), overwrite=True)

    # [D] Save var_names.csv for SCANVI model -- the gene set is identical to scVI
    # but storing it alongside the SCANVI directory removes dependency on scVI path
    # when a query pipeline loads SCANVI directly via load_query_data().
    pd.Series(adata.var_names.tolist()).to_csv(
        SCANVI_MODEL_PATH / "var_names.csv", index=False, header=False
    )
    print(f"[OK] SCANVI var_names.csv saved -> {SCANVI_MODEL_PATH/'var_names.csv'}")

    adata.obsm['X_scanvi']             = model_scanvi.get_latent_representation()
    adata.obs['cell_type_scanvi_pred'] = model_scanvi.predict()
    soft_pred     = model_scanvi.predict(soft=True)
    soft_pred_arr = soft_pred.to_numpy() if hasattr(soft_pred, 'to_numpy') else np.asarray(soft_pred)
    adata.obs['scanvi_uncertainty']    = (1 - soft_pred_arr.max(axis=1)).astype(np.float32)
    print(f"[OK] X_scanvi: {adata.obsm['X_scanvi'].shape}")

    # [C] scArches dry-run validation for SCANVI
    print("\n[scArches validation] Running SCANVI prepare_query_anndata() dry-run (2k subset)...")
    try:
        # [P2-6] Use 2k-cell subset to avoid duplicating full adata in RAM
        _n_check = min(2000, adata.n_obs)
        adata_check = adata[:_n_check, :].copy()
        scvi.model.SCANVI.prepare_query_anndata(adata_check, str(SCANVI_MODEL_PATH))
        del adata_check
        gc.collect()
        print("[OK] SCANVI model passes scArches prepare_query_anndata() validation")
    except Exception as e:
        print(f"[WARN] SCANVI prepare_query_anndata() raised: {e}")

    print("scANVI prediction distribution:")
    print(adata.obs['cell_type_scanvi_pred'].value_counts().head(25).to_string())



STEP 11: scANVI TRAINING (scArches-compatible)
Labeled classes (excluding 'Unknown'): 36
[OK] GPU cache cleared before scANVI training
Training backend: accelerator=gpu, devices=1
INFO     Training for 200 epochs.                                                                                  


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 200/200: 100%|██████████| 200/200 [40:43<00:00, 12.22s/it, v_num=1, train_loss_step=682, train_loss_epoch=738]

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 200/200: 100%|██████████| 200/200 [40:43<00:00, 12.22s/it, v_num=1, train_loss_step=682, train_loss_epoch=738]
[OK] scANVI done in 40.7 min
[OK] SCANVI var_names.csv saved -> /home/h2048/data/py/0308/stromal_reintegration_v1_2/models/scanvi_stromal_v1/var_names.csv
[OK] X_scanvi: (56689, 75)

[scArches validation] Running SCANVI prepare_query_anndata() dry-run (2k subset)...
INFO     File /home/h2048/data/py/0308/stromal_reintegration_v1_2/models/scanvi_stromal_v1/model.pt already        
         downloaded                                                                                                
INFO     Found 100.0% reference vars in query data.                                                                
[OK] SCANVI model passes scArches prepare_query_anndata() validation
scANVI prediction distribution:
cell_type_scanvi_pred
Activated Antigen-presenting Venous Endothelia               5747
Immune-recruiting (ACKR1+SELE+) Venous Endothelia            5352
Unresolved Ca

## Step 12: UMAP + FIGURES (scANVI latent)

In [13]:
# ============================================================================
# STEP 12: UMAP + FIGURES (scANVI latent)
# ============================================================================
print("\n" + "="*70 + "\nSTEP 12: UMAP (scANVI) + FIGURES\n" + "="*70)
sc.pp.neighbors(adata, use_rep='X_scanvi', n_neighbors=30,
                random_state=RANDOM_SEED, key_added='neighbors_scanvi')
sc.tl.umap(adata, neighbors_key='neighbors_scanvi', random_state=RANDOM_SEED)
adata.obsm['X_umap_scanvi'] = adata.obsm['X_umap'].copy()

fig, axes = plt.subplots(1, 3, figsize=(27, 8))
sc.pl.embedding(adata, basis='umap', color=CELLTYPE_L2, ax=axes[0],
                show=False, frameon=False, size=2, legend_loc='right margin',
                legend_fontsize=7, title='L2 Major Type')
sc.pl.embedding(adata, basis='umap', color='scanvi_label', ax=axes[1],
                show=False, frameon=False, size=2, legend_loc='right margin',
                legend_fontsize=6, title='scANVI Training Label (coarse_L3)')
sc.pl.embedding(adata, basis='umap', color='cell_type_scanvi_pred', ax=axes[2],
                show=False, frameon=False, size=2, legend_loc='right margin',
                legend_fontsize=6, title='scANVI Prediction')
plt.suptitle('Stromal/Vascular Post-scANVI', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_DIR/'scanvi_umap_overview.pdf', dpi=300, bbox_inches='tight')
plt.close('all'); print("[OK] Saved: scanvi_umap_overview.pdf")

fig, ax = plt.subplots(figsize=(8, 7))
sc.pl.embedding(adata, basis='umap', color='scanvi_uncertainty',
                ax=ax, show=False, frameon=False, size=2,
                cmap='RdYlBu_r', vmin=0, vmax=0.5,
                title='scANVI Prediction Uncertainty')
fig.savefig(FIG_DIR/'scanvi_uncertainty_umap.pdf', dpi=300, bbox_inches='tight')
plt.close('all'); print("[OK] Saved: scanvi_uncertainty_umap.pdf")

if TISSUE_KEY_AVAILABLE:
    fig, ax = plt.subplots(figsize=(9, 7))
    sc.pl.embedding(adata, basis='umap', color=TISSUE_KEY, ax=ax,
                    show=False, frameon=False, size=2, title='Tissue Source')
    fig.savefig(FIG_DIR/'scanvi_umap_tissue.pdf', dpi=300, bbox_inches='tight')
    plt.close('all'); print("[OK] Saved: scanvi_umap_tissue.pdf")
gc.collect()



STEP 12: UMAP (scANVI) + FIGURES
computing neighbors
    finished (0:00:13)
computing UMAP
    finished (0:01:44)
[OK] Saved: scanvi_umap_overview.pdf
[OK] Saved: scanvi_uncertainty_umap.pdf
[OK] Saved: scanvi_umap_tissue.pdf


12945

## Step 13: LEIDEN CLUSTERING (scANVI latent)

In [14]:
# ============================================================================
# STEP 13: LEIDEN CLUSTERING (scANVI latent)
# ============================================================================
print("\n" + "="*70 + "\nSTEP 13: LEIDEN CLUSTERING (scANVI latent)\n" + "="*70)
for res in [0.4, 0.6, 0.8, 1.0]:
    key = f'leiden_scanvi_r{res:.1f}'
    sc.tl.leiden(adata, resolution=res, neighbors_key='neighbors_scanvi',
                 key_added=key, random_state=RANDOM_SEED)
    print(f"  Resolution {res}: {adata.obs[key].nunique()} clusters")



STEP 13: LEIDEN CLUSTERING (scANVI latent)
running Leiden clustering
    finished (0:01:00)
  Resolution 0.4: 13 clusters
running Leiden clustering
    finished (0:00:58)
  Resolution 0.6: 16 clusters
running Leiden clustering
    finished (0:00:45)
  Resolution 0.8: 18 clusters
running Leiden clustering
    finished (0:01:12)
  Resolution 1.0: 21 clusters


## Step 14: SAVE FINAL OUTPUT + reference_manifest.json

In [15]:
# ============================================================================
# STEP 14: SAVE OUTPUT + reference_manifest.json
# ============================================================================
# [E] reference_manifest.json is a machine-readable descriptor consumed by the
#     scArches query pipeline (Step 15 template) to locate model paths, gene
#     lists, label columns, and batch keys without hardcoding paths in query scripts.
# ============================================================================
print("\n" + "="*70 + "\nSTEP 14: SAVE OUTPUT\n" + "="*70)

params = {
    'version': '1.3',
    'date': '2026-03-08',
    'n_cells_input': int(n_before),
    'n_cells_output': int(adata.n_obs),
    'drop_clusters': DROP_CLUSTERS,
    'reassign_fibro': ['Endothelia_vascular_venous_systemic_c4'],
    'reassign_alveolar': ['Fibro_adventitial_c2'],
    'n_hvg': int(N_HVG),
    'hvg_method': hvg_method,
    'n_latent': N_LATENT,
    'encode_covariates': True,
    'batch_key': BATCH_KEY,
    'tissue_key': TISSUE_KEY,
    'celltype_l3_column_used': CELLTYPE_L3,
    'tissue_key_available': TISSUE_KEY_AVAILABLE,
    'lung_trachea_values': sorted(LUNG_TRACHEA_VALUES),
    'pulmonary_alveolar_clusters': PULMONARY_ALVEOLAR_CLUSTERS,
    'purity_k': PURITY_K,
    'purity_threshold': PURITY_THRESHOLD,
    'random_seed': RANDOM_SEED,
    'fixes': [
        'scanvi_setup_anndata_removed',
        'knn_purity_vectorized',
        'scvi_model_released_before_scanvi',
        'counts_fallback_from_raw',
        'encode_covariates_added_for_scarches',
        'prepare_query_anndata_validation_added',
        'scanvi_var_names_csv_saved',
        'reference_manifest_saved',
    ],
}
adata.uns['stromal_reintegration_params'] = params

print(f"Cells: {adata.n_obs:,} | HVGs: {adata.n_vars:,} | .raw: {adata.raw.n_vars:,}")
adata.write_h5ad(OUTPUT_H5AD, compression='gzip', compression_opts=9)
pd.Series(adata.var_names.tolist()).to_csv(OUTPUT_DIR/'hvg_genes_final.csv', index=False, header=False)
pd.Series(adata.raw.var_names.tolist()).to_csv(OUTPUT_DIR/'all_genes_raw.csv', index=False, header=False)
adata.obs[[
    CELLTYPE_L2, CELLTYPE_L3, 'coarse_L3', 'annot_action', 'annot_confidence',
    'scanvi_label', 'label_purity_scanvi', 'cell_type_scanvi_pred', 'scanvi_uncertainty'
]].to_csv(OUTPUT_DIR/'scanvi_label_summary.csv')

# [E] Save reference manifest for scArches query pipelines
# [P1-4] SCANVI_TRAINED guards: manifest truthfully reflects whether scANVI was trained
manifest = {
    'reference_h5ad': str(OUTPUT_H5AD),
    'scvi_model_dir': str(SCVI_MODEL_PATH),
    'scanvi_model_dir': str(SCANVI_MODEL_PATH) if SCANVI_TRAINED else None,
    'scanvi_available': bool(SCANVI_TRAINED),
    'hvg_var_names_csv': str(SCVI_MODEL_PATH / 'var_names.csv'),
    'all_genes_csv': str(OUTPUT_DIR / 'all_genes_raw.csv'),
    'batch_key': BATCH_KEY,
    'label_key': 'scanvi_label',
    'prediction_key': 'cell_type_scanvi_pred',
    'uncertainty_key': 'scanvi_uncertainty',
    'unlabeled_category': UNLABELED,
    'n_latent': N_LATENT,
    'encode_covariates': True,
    'version': '1.3',
    'date': '2026-03-08',
    'scarches_compatible': True,
}
manifest_path = OUTPUT_DIR / 'reference_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f"[OK] reference_manifest.json saved -> {manifest_path}")
print(f"[OK] {OUTPUT_H5AD}")

elapsed = time.time() - PIPELINE_START
print(f"\n{'='*70}\nPIPELINE COMPLETE  ({elapsed/60:.1f} min)\n{'='*70}")
print(f"Cells: {n_before:,} -> {adata.n_obs:,}  |  "
      f"Labeled: {n_labeled:,}  |  Unknown: {n_unknown:,}")
print(f"\nNext steps:")
print(f"  1. Confirm TISSUE_KEY / LUNG_TRACHEA_VALUES match your data")
print(f"  2. Review scanvi_umap_overview.pdf -- label vs prediction agreement")
print(f"  3. Check scanvi_label_summary.csv for high-uncertainty REVIEW clusters")
print(f"  4. Use reference_manifest.json with the Step 15 query template for new cohorts")



STEP 14: SAVE OUTPUT
Cells: 56,689 | HVGs: 4,000 | .raw: 36,789
[OK] reference_manifest.json saved -> /home/h2048/data/py/0308/stromal_reintegration_v1_2/reference_manifest.json
[OK] /home/h2048/data/py/0308/stromal_reintegration_v1_2/stromal_reintegrated_scvi_scanvi_v1_2.h5ad

PIPELINE COMPLETE  (97.7 min)
Cells: 60,828 -> 56,689  |  Labeled: 26,765  |  Unknown: 29,924

Next steps:
  1. Confirm TISSUE_KEY / LUNG_TRACHEA_VALUES match your data
  2. Review scanvi_umap_overview.pdf -- label vs prediction agreement
  3. Check scanvi_label_summary.csv for high-uncertainty REVIEW clusters
  4. Use reference_manifest.json with the Step 15 query template for new cohorts


## Step 15: RUN LOG + AnnData STRUCTURE SUMMARY

In [16]:
# ============================================================================
# STEP 15: RUN LOG + AnnData STRUCTURE SUMMARY
# ============================================================================
# Mirrors the _write_branch_logs() pattern from the epithelial pipeline.
# Writes two plain-text files to OUTPUT_DIR:
#   pipeline_run_log.txt      -- key parameters, cell counts, elapsed time,
#                                prediction distributions
#   anndata_structure.txt     -- full inventory of .X / .obs / .var / .layers /
#                                .obsm / .obsp / .uns / .raw
# ============================================================================
from datetime import datetime

print("\n" + "=" * 80)
print("STEP 15: Writing Run Log and AnnData Structure Summary")
print("=" * 80)

# ---------- helpers (identical style to epithelial pipeline) ----------------- #

def _fmt_shape(obj):
    shape = getattr(obj, 'shape', None)
    return 'NA' if shape is None else ' x '.join(str(x) for x in shape)

def _matrix_line(name, obj):
    return (
        f"- {name}: type={type(obj).__name__}, shape={_fmt_shape(obj)}, "
        f"dtype={getattr(obj, 'dtype', 'NA')}, "
        f"sparse={sparse.issparse(obj) if obj is not None else False}"
    )

def _mapping_lines(title, mapping):
    lines = [f"{title} ({len(mapping)}):"]
    if len(mapping) == 0:
        lines.append("  (none)")
        return lines
    for key, value in mapping.items():
        lines.append(
            f"  - {key}: type={type(value).__name__}, "
            f"shape={getattr(value, 'shape', 'NA')}, "
            f"dtype={getattr(value, 'dtype', 'NA')}"
        )
    return lines

def _df_column_lines(df, title):
    lines = [f"{title} ({len(df.columns)} columns):"]
    if len(df.columns) == 0:
        lines.append("  (none)")
        return lines
    for col in df.columns:
        lines.append(
            f"  - {col}: dtype={df[col].dtype}, "
            f"non_null={df[col].notna().sum():,}, "
            f"unique={df[col].nunique(dropna=False):,}"
        )
    return lines

# ---------- write run log ---------------------------------------------------- #

elapsed_min = (time.time() - PIPELINE_START) / 60

run_lines = [
    "=" * 80,
    "Stromal/Vascular Reintegration Pipeline Run Log — v1.2 (scArches-Compatible)",
    "=" * 80,
    f"written_at:                {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"input_h5ad:                {INPUT_H5AD}",
    f"output_h5ad:               {OUTPUT_H5AD}",
    f"output_h5ad_exists:        {OUTPUT_H5AD.exists()}",
    "",
    "--- Cell counts ---",
    f"n_cells_input:             {n_before:,}",
    f"n_cells_output:            {adata.n_obs:,}",
    f"n_cells_removed:           {n_before - adata.n_obs:,}",
    f"n_batches:                 {adata.obs[BATCH_KEY].nunique():,}",
    f"n_hvg:                     {adata.n_vars:,}",
    f"n_genes_full (.raw):       {adata.raw.n_vars:,}",
    f"hvg_method:                {hvg_method}",
    "",
    "--- scVI / scANVI ---",
    f"n_latent:                  {N_LATENT}",
    f"encode_covariates:         True  (scArches-compatible)",
    f"purity_k:                  {PURITY_K}",
    f"purity_threshold:          {PURITY_THRESHOLD}",
    f"n_labeled:                 {n_labeled:,}",
    f"n_unknown:                 {n_unknown:,}",
    f"pct_unknown:               {n_unknown / adata.n_obs * 100:.1f}%",
    "",
    "--- Paths ---",
    f"scvi_model_dir:            {MODEL_DIR / 'scvi_stromal_v1'}",
    f"scanvi_model_dir:          {MODEL_DIR / 'scanvi_stromal_v1'}",
    f"figure_dir:                {FIG_DIR}",
    f"reference_manifest:        {OUTPUT_DIR / 'reference_manifest.json'}",
    f"elapsed_minutes:           {elapsed_min:.2f}",
]

if OUTPUT_H5AD.exists():
    run_lines.append(f"output_h5ad_size_gb:       {OUTPUT_H5AD.stat().st_size / 1e9:.3f}")

# Drop clusters summary
run_lines += [
    "",
    "--- Contamination removal ---",
    f"drop_clusters ({len(DROP_CLUSTERS)}): {DROP_CLUSTERS}",
]

# Prediction distributions
run_lines += [
    "",
    "scanvi_label distribution (training labels):",
    adata.obs['scanvi_label'].value_counts().to_string(),
    "",
    "cell_type_scanvi_pred distribution (final predictions):",
    adata.obs['cell_type_scanvi_pred'].value_counts().to_string(),
    "",
    f"scanvi_uncertainty -- mean: {adata.obs['scanvi_uncertainty'].mean():.4f}, "
    f"median: {adata.obs['scanvi_uncertainty'].median():.4f}, "
    f"pct>0.3: {(adata.obs['scanvi_uncertainty'] > 0.3).sum() / adata.n_obs * 100:.1f}%",
]

# Leiden cluster counts
for col in [c for c in adata.obs.columns if c.startswith('leiden_scanvi_r')]:
    run_lines.append(f"{col}: {adata.obs[col].nunique()} clusters")

run_log_path = OUTPUT_DIR / "pipeline_run_log.txt"
run_log_path.write_text('\n'.join(run_lines) + '\n', encoding='utf-8')
print(f"[OK] {run_log_path.name}")

# ---------- write AnnData structure ------------------------------------------ #

structure_lines = [
    "=" * 80,
    "AnnData Structure Summary — Stromal/Vascular Reintegration v1.2",
    "=" * 80,
    f"written_at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"shape: {adata.shape}",
    _matrix_line('X', adata.X),
    "",
]
structure_lines.extend(_df_column_lines(adata.obs, 'obs'))
structure_lines.append("")
structure_lines.extend(_df_column_lines(adata.var, 'var'))
structure_lines.append("")
structure_lines.extend(_mapping_lines('layers', adata.layers))
structure_lines.append("")
structure_lines.extend(_mapping_lines('obsm', adata.obsm))
structure_lines.append("")
structure_lines.extend(_mapping_lines('varm', adata.varm))
structure_lines.append("")
structure_lines.extend(_mapping_lines('obsp', adata.obsp))
structure_lines.append("")
structure_lines.extend(_mapping_lines('varp', adata.varp))
structure_lines.append("")
structure_lines.extend(_mapping_lines('uns', adata.uns))
structure_lines.append("")

if adata.raw is not None:
    structure_lines.append(f"raw: shape={adata.raw.shape}")
    raw_var_cols = ', '.join(adata.raw.var.columns.astype(str)) if len(adata.raw.var.columns) else '(none)'
    structure_lines.append(f"raw.var columns: {raw_var_cols}")
else:
    structure_lines.append("raw: None")

structure_path = OUTPUT_DIR / "anndata_structure.txt"
structure_path.write_text('\n'.join(structure_lines) + '\n', encoding='utf-8')
print(f"[OK] {structure_path.name}")

print(f"\n[OK] Step 15 complete -- logs written to {OUTPUT_DIR}")



STEP 15: Writing Run Log and AnnData Structure Summary
[OK] pipeline_run_log.txt
[OK] anndata_structure.txt

[OK] Step 15 complete -- logs written to /home/h2048/data/py/0308/stromal_reintegration_v1_2


## Step 16: scArches QUERY MAPPING TEMPLATE (copy-paste for new cohorts)

In [18]:
# ============================================================================
# STEP 15: scArches QUERY MAPPING TEMPLATE
# ============================================================================
# This cell is a STANDALONE template -- copy it to a new notebook when you have
# a new cohort / center to annotate against the stromal reference trained above.
#
# Requirements:
#   - adata_query must have layer='counts' (raw counts) + obs[QUERY_BATCH_KEY]
#   - Gene overlap with reference HVGs must be >= 80%
#   - scvi-tools >= 1.1  (provides load_query_data and prepare_query_anndata)
# ============================================================================

import json, gc, numpy as np, pandas as pd
import scanpy as sc, scvi
from scipy import sparse
from pathlib import Path
from datetime import datetime

# ---------- USER CONFIG ----------------------------------------------------- #
QUERY_H5AD       = Path("/path/to/new_query_stromal.h5ad")
MANIFEST_PATH    = Path("/home/h2048/data/py/0308/stromal_reintegration_v1_2/reference_manifest.json")
QUERY_OUTPUT_DIR = Path("/home/h2048/data/py/YYYYMMDD/stromal_query_mapping")
QUERY_BATCH_KEY  = 'sample'   # batch column in query adata.obs
MAX_EPOCHS_QUERY_SCVI   = 200  # fine-tuning epochs (fewer than full training)
MAX_EPOCHS_QUERY_SCANVI = 100
WEIGHT_DECAY     = 0.0
MIN_GENE_OVERLAP = 0.80
RANDOM_SEED      = 42
# ---------------------------------------------------------------------------- #

QUERY_H5AD_PLACEHOLDER = Path("/path/to/new_query_stromal.h5ad")
QUERY_OUTPUT_DIR_PLACEHOLDER = "YYYYMMDD"

if QUERY_OUTPUT_DIR_PLACEHOLDER in str(QUERY_OUTPUT_DIR):
    QUERY_OUTPUT_DIR = (
        Path("/home/h2048/data/py")
        / datetime.now().strftime("%Y%m%d")
        / "stromal_query_mapping"
    )
    print(f"[INFO] QUERY_OUTPUT_DIR placeholder detected; using {QUERY_OUTPUT_DIR}")

USE_GPU = __import__('torch').cuda.is_available()

# ── Load manifest ────────────────────────────────────────────────────────────
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"[ERROR] MANIFEST_PATH not found: {MANIFEST_PATH}")

with open(MANIFEST_PATH) as f:
    ref = json.load(f)
SCVI_MODEL_PATH  = Path(ref['scvi_model_dir'])
SCANVI_MODEL_PATH = Path(ref['scanvi_model_dir'])
UNLABELED        = ref['unlabeled_category']
BATCH_KEY_REF    = ref['batch_key']

# Reference HVG gene list
ref_genes = pd.read_csv(ref['hvg_var_names_csv'], header=None)[0].tolist()
print(f"Reference HVGs: {len(ref_genes)}")

query_path_is_placeholder = (
    QUERY_H5AD == QUERY_H5AD_PLACEHOLDER
    or str(QUERY_H5AD).startswith("/path/to/")
)
query_path_exists = QUERY_H5AD.exists()

if query_path_is_placeholder or not query_path_exists:
    print("[SKIP] QUERY_H5AD is still a template path or the file does not exist.")
    print(f"  Current QUERY_H5AD: {QUERY_H5AD}")
    print("  Please update QUERY_H5AD to a real stromal query .h5ad before running this step.")

    candidate_root = Path("/home/h2048/data/py")
    candidate_query_files = sorted(
        p for p in candidate_root.rglob("*stromal*.h5ad") if p.is_file()
    )
    if candidate_query_files:
        print("\n  Candidate stromal .h5ad files under /home/h2048/data/py:")
        for p in candidate_query_files[:10]:
            print(f"    - {p}")
        if len(candidate_query_files) > 10:
            print(f"    ... and {len(candidate_query_files) - 10} more")
    else:
        print("\n  [INFO] No stromal .h5ad candidates were found under /home/h2048/data/py")

    print("\n  Example:")
    print("    QUERY_H5AD = Path('/home/h2048/data/py/<DATE>/<your_query>.h5ad')")
    print("[DONE] Step 15 skipped until QUERY_H5AD is configured.")
else:
    QUERY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # ── Load + align query ───────────────────────────────────────────────────
    adata_query = sc.read_h5ad(QUERY_H5AD)
    assert 'counts' in adata_query.layers, "[ERROR] adata_query missing layers['counts']"
    print(f"Query: {adata_query.n_obs:,} cells x {adata_query.n_vars:,} genes")

    query_genes = set(adata_query.var_names)
    ref_genes_set = set(ref_genes)
    overlap = query_genes & ref_genes_set
    overlap_rate = len(overlap) / len(ref_genes_set)
    print(f"Gene overlap: {len(overlap)}/{len(ref_genes_set)} = {overlap_rate*100:.1f}%")
    if overlap_rate < MIN_GENE_OVERLAP:
        raise ValueError(f"Gene overlap {overlap_rate*100:.1f}% < {MIN_GENE_OVERLAP*100}% threshold")

    # Add zero columns for missing genes (scArches requirement)
    missing = ref_genes_set - query_genes
    if missing:
        print(f"[INFO] Padding {len(missing)} missing genes with zeros")
        zero_mat = sparse.csr_matrix(
            (adata_query.n_obs, len(missing)), dtype=np.float32
        )
        missing_adata = sc.AnnData(
            X=zero_mat,
            obs=adata_query.obs.copy(),
            var=pd.DataFrame(index=sorted(missing))
        )
        # Store counts layer in missing_adata too
        missing_adata.layers['counts'] = zero_mat.copy()
        adata_query = sc.concat([adata_query, missing_adata], axis=1,
                                 join='outer', merge='same')

    # Reorder to exact reference gene order
    adata_query = adata_query[:, ref_genes].copy()
    if 'counts' not in adata_query.layers:
        adata_query.layers['counts'] = adata_query.X.copy()
    # [P0-2] The inplace=False call below was removed: it returned a new object
    # without assigning it, so it had zero effect and misled the reader.
    # Correct normalization sequence:
    adata_query.X = adata_query.layers['counts'].copy()
    sc.pp.normalize_total(adata_query, target_sum=1e4)
    sc.pp.log1p(adata_query)
    print(f"[OK] Query aligned: {adata_query.shape}")

    # ── scVI architecture surgery ────────────────────────────────────────────
    # [P0-1] Rename query batch key to match reference BEFORE prepare_query_anndata.
    # prepare_query_anndata aligns the data manager registry; if the reference batch
    # column does not exist yet at that point, the registry validation will fail or
    # silently fall back to wrong behavior.
    if QUERY_BATCH_KEY not in adata_query.obs.columns:
        raise KeyError(f"[ERROR] Query batch key not found: '{QUERY_BATCH_KEY}'")

    if QUERY_BATCH_KEY != BATCH_KEY_REF:
        adata_query.obs[BATCH_KEY_REF] = adata_query.obs[QUERY_BATCH_KEY].astype(str).values

    adata_query.obs[BATCH_KEY_REF] = adata_query.obs[BATCH_KEY_REF].astype('category')

    print("\n[scArches] Preparing query for scVI surgery...")
    scvi.model.SCVI.prepare_query_anndata(adata_query, str(SCVI_MODEL_PATH))

    query_model_scvi = scvi.model.SCVI.load_query_data(
        adata_query,
        str(SCVI_MODEL_PATH),
    )
    print("[OK] Query scVI model created via architecture surgery")

    train_kwargs = dict(
        max_epochs=MAX_EPOCHS_QUERY_SCVI,
        plan_kwargs={'weight_decay': WEIGHT_DECAY},
        accelerator='gpu' if USE_GPU else 'cpu',
        devices=1,
    )
    import time; t0 = time.time()
    query_model_scvi.train(**train_kwargs)
    print(f"[OK] Query scVI fine-tuning done in {(time.time()-t0)/60:.1f} min")
    query_model_scvi.save(str(QUERY_OUTPUT_DIR / "query_scvi_model"), overwrite=True)
    adata_query.obsm['X_scvi'] = query_model_scvi.get_latent_representation()

    # ── scANVI architecture surgery (label transfer) ─────────────────────────
    # [P1-4] Skip scANVI surgery if no trained scANVI reference exists
    if not ref.get('scanvi_available', True):
        print("[WARN] manifest scanvi_available=False -- scANVI label transfer skipped.")
        adata_query.obsm['X_scanvi']             = adata_query.obsm['X_scvi'].copy()
        adata_query.obs['cell_type_scanvi_pred'] = UNLABELED
        adata_query.obs['scanvi_uncertainty']    = np.full(adata_query.n_obs, 1.0, dtype=np.float32)
    else:
        # Label all query cells as Unknown -- scANVI will predict labels
        adata_query.obs['scanvi_label'] = UNLABELED

        scvi.model.SCANVI.prepare_query_anndata(adata_query, str(SCANVI_MODEL_PATH))

        query_model_scanvi = scvi.model.SCANVI.load_query_data(
            adata_query,
            str(SCANVI_MODEL_PATH),
        )
        print("[OK] Query scANVI model created via architecture surgery")

        t0 = time.time()
        query_model_scanvi.train(
            max_epochs=MAX_EPOCHS_QUERY_SCANVI,
            plan_kwargs={'weight_decay': WEIGHT_DECAY},
            accelerator='gpu' if USE_GPU else 'cpu',
            devices=1,
        )
        print(f"[OK] Query scANVI fine-tuning done in {(time.time()-t0)/60:.1f} min")
        query_model_scanvi.save(str(QUERY_OUTPUT_DIR / 'query_scanvi_model'), overwrite=True)

        adata_query.obsm['X_scanvi']             = query_model_scanvi.get_latent_representation()
        adata_query.obs['cell_type_scanvi_pred'] = query_model_scanvi.predict()
        soft_pred     = query_model_scanvi.predict(soft=True)
        soft_pred_arr = soft_pred.to_numpy() if hasattr(soft_pred, 'to_numpy') else np.asarray(soft_pred)
        adata_query.obs['scanvi_uncertainty']    = (1 - soft_pred_arr.max(axis=1)).astype(np.float32)
        print("[OK] Label transfer complete")

    # ── UMAP + save ──────────────────────────────────────────────────────────
    sc.pp.neighbors(adata_query, use_rep='X_scanvi', n_neighbors=30, random_state=RANDOM_SEED)
    sc.tl.umap(adata_query, random_state=RANDOM_SEED)

    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    sc.pl.umap(adata_query, color='cell_type_scanvi_pred', ax=axes[0],
               show=False, frameon=False, size=3, title='Transferred Labels')
    sc.pl.umap(adata_query, color='scanvi_uncertainty', cmap='RdYlBu_r',
               vmin=0, vmax=0.5, ax=axes[1], show=False, frameon=False,
               title='Prediction Uncertainty')
    plt.tight_layout()
    fig.savefig(QUERY_OUTPUT_DIR / 'query_scarches_umap.pdf', dpi=300, bbox_inches='tight')
    plt.close('all')

    # Confidence summary
    low_conf_pct = (adata_query.obs['scanvi_uncertainty'] > 0.3).sum() / adata_query.n_obs * 100
    print(f"\nLabel transfer quality:")
    print(f"  High uncertainty (>0.3): {low_conf_pct:.1f}%")
    if low_conf_pct > 30:
        print("[WARN] >30% cells high uncertainty -- check gene overlap and batch alignment")

    print("\nPredicted label distribution:")
    print(adata_query.obs['cell_type_scanvi_pred'].value_counts().head(20).to_string())

    adata_query.write_h5ad(
        QUERY_OUTPUT_DIR / 'stromal_query_mapped.h5ad',
        compression='gzip', compression_opts=9
    )
    print(f"\n[OK] Saved: {QUERY_OUTPUT_DIR / 'stromal_query_mapped.h5ad'}")
    print("[DONE] scArches query mapping complete.")

[INFO] QUERY_OUTPUT_DIR placeholder detected; using /home/h2048/data/py/20260309/stromal_query_mapping
Reference HVGs: 4000
[SKIP] QUERY_H5AD is still a template path or the file does not exist.
  Current QUERY_H5AD: /path/to/new_query_stromal.h5ad
  Please update QUERY_H5AD to a real stromal query .h5ad before running this step.

  Candidate stromal .h5ad files under /home/h2048/data/py:
    - /home/h2048/data/py/0111/celltypist_stromal/adata_stromal_FINAL.h5ad
    - /home/h2048/data/py/0114/stromal_pure_scanvi/adata_stromal_PURE_FINAL.h5ad
    - /home/h2048/data/py/0120/stromal_analysis_unified/results/subcluster_unified_v2_20260128/adata_stromal_subclustered_FINAL_v2_20260128.h5ad
    - /home/h2048/data/py/0303/stromal_reintegration_v1/stromal_reintegrated_scvi_scanvi_v1_1.h5ad
    - /home/h2048/data/py/0308/stromal_reintegration_v1_2/stromal_reintegrated_scvi_scanvi_v1_2.h5ad
    - /home/h2048/data/py/1212/stromal_vascular_celltypist_v2_1_production/adata_stromal_vascular_FINAL.h5a